In [ ]:
# hide
import numpy as np
import pyquist as pq
from pyquist.helper import pitch_name_to_pitch

sax = {name: pq.Audio.from_file(f"../assets/sax-{name}.wav") for name in ("C4", "C5", "C6")}


def cut_off(x: pq.Audio, duration: float, r_dur: float = 0.1) -> pq.Audio:
    """End a note early: keep `duration` seconds, fading out over the last r_dur."""
    y = np.asarray(x.samples)[:int(duration * x.sample_rate)].copy()
    n_r = min(len(y), int(r_dur * x.sample_rate))
    y[len(y) - n_r:] *= np.linspace(1.0, 0.0, n_r)[:, np.newaxis]
    return pq.Audio(y, x.sample_rate)

In [ ]:
def resample(x: pq.Audio, ratio: float) -> pq.Audio:
    """Read x at `ratio` times its original speed, with linear interpolation."""
    samples = np.asarray(x.samples)
    N = len(samples)
    M = int(N / ratio)                        # the duration scales by 1 / ratio
    p = np.arange(M) * ratio                  # fractional read positions in x
    p_floor = np.floor(p).astype(int)
    alpha = (p - p_floor)[:, np.newaxis]
    p_next = np.minimum(p_floor + 1, N - 1)
    y = (1 - alpha) * samples[p_floor] + alpha * samples[p_next]
    return pq.Audio(y, x.sample_rate)


def repitch(x: pq.Audio, k: float) -> pq.Audio:
    """Shift the pitch of x by k steps."""
    return resample(x, 2 ** (k / 12))


# Pitch and duration are coupled: repitching the C5 recording also changes its duration
for name in ("C4", "C5", "C6"):
    k = pitch_name_to_pitch(name) - pitch_name_to_pitch("C5")
    print(f"C5 recording repitched to {name} (k = {k:+d}): {repitch(sax['C5'], k).duration:.2f} s")

In [ ]:
def from_c5(pitch: str, duration: float) -> pq.Audio:
    """Repitch the single C5 recording to any pitch."""
    k = pitch_name_to_pitch(pitch) - pitch_name_to_pitch("C5")
    return cut_off(repitch(sax["C5"], k), duration)


def from_nearest_c(pitch: str, duration: float) -> pq.Audio:
    """Repitch whichever recording (C4, C5, or C6) is closest to the target pitch."""
    nearest = min(sax, key=lambda c: abs(pitch_name_to_pitch(c) - pitch_name_to_pitch(pitch)))
    k = pitch_name_to_pitch(pitch) - pitch_name_to_pitch(nearest)
    return cut_off(repitch(sax[nearest], k), duration)


arpeggio = pq.Score([(0.45 * i, {"pitch": pitch, "duration": 0.45})
                     for i, pitch in enumerate(["C4", "E4", "G4", "C5", "E5", "G5", "C6"])])
pq.play(arpeggio.render(from_c5), normalize=True)
pq.play(arpeggio.render(from_nearest_c), normalize=True)